# Параллельный MCTS и AlphaZero на GPU
В меню **Среда выполнения → Сменить среду выполнения** выберите GPU. Подготовьте архив `parallel-alphazero-colab.zip` командой из README. Этот ноутбук запускает исходный код проекта, сохраняет реальные измерения и контрольные точки.
Используется расширенный профиль обучения «Четыре в ряд»: 50 итераций × 24 партии, 64 симуляции на ход и 80 обновлений сети на итерацию. По предыдущему запуску ориентир — около 25 минут обучения, не гарантия. Итоговое сравнение методов MCTS выполняется с прежним полным бюджетом. Сначала выполните тесты и короткую проверку; затем обучение и измерения. Не останавливайте среду до сохранения `latest.pt`.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
archive_name = next(name for name in uploaded if name.endswith(".zip"))
project = Path("/content/parallel-alphazero")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_name) as archive:
    for member in archive.infolist():
        target = (project / member.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(project)
os.chdir(project)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy", "matplotlib", "pytest"], check=True
)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Включите GPU в настройках Colab"
print(torch.cuda.get_device_name(0))
print("Muon available:", hasattr(torch.optim, "Muon"))
# Детерминированные матричные операции CUDA требуют этой настройки.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "azparallel", "doctor", "--device", "cuda"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## Сохранение на Drive
Каталоги результатов находятся на вашем Drive; исходный код работает в `/content`. По умолчанию создаётся новая папка. Чтобы продолжить прошлое обучение, заполните `resume_run_name` именем прежней папки (например, `20261014_142856`). Старый каталог `training_gpu` не изменяется: его копия продолжается в `training_gpu_50`. При восстановлении пропустите уже завершённую короткую проверку. Частично завершённые измерения повторяйте в новом каталоге.


In [ ]:
from datetime import UTC, datetime

from google.colab import drive

drive.mount("/content/drive")
resume_run_name = ""  # Для продолжения: имя прежней папки запуска на Drive.
run_name = resume_run_name or datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
output = Path("/content/drive/MyDrive/parallel-alphazero") / run_name
if resume_run_name and not output.is_dir():
    raise FileNotFoundError(f"Прежняя папка не найдена: {output}")
output.mkdir(parents=True, exist_ok=True)
print(output)


def run(*args):
    command = [sys.executable, "-u", "-m", "azparallel", *map(str, args)]
    with subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

In [ ]:
# Проверка именно GPU-пути, очереди и обучения на коротком запуске.
smoke = json.loads(Path("configs/train_colab.json").read_text())
smoke.update(
    iterations=1,
    games_per_iteration=2,
    updates_per_iteration=2,
    simulations=8,
    eval_games=2,
    eval_positions=4,
    eval_every=1,
)
smoke_path = output / "smoke_config.json"
smoke_path.write_text(json.dumps(smoke), encoding="utf-8")
run("train", "--config", smoke_path, "--device", "cuda", "--out", output / "smoke_50")
run("inference", "--config", "configs/inference_colab.json", "--out", output / "inference_gpu_50")

In [ ]:
# До 50 итераций; старая модель и измерения сохраняются отдельно.
train_dir = output / "training_gpu_50"
previous_train_dir = output / "training_gpu"
if not train_dir.exists() and (previous_train_dir / "latest.pt").is_file():
    shutil.copytree(previous_train_dir, train_dir)
    print("Копия предыдущего обучения создана:", train_dir)
args = ["train", "--config", "configs/train_colab.json", "--device", "cuda", "--out", train_dir]
if (train_dir / "latest.pt").exists():
    args += ["--resume", train_dir / "latest.pt"]
run(*args)
saved = torch.load(train_dir / "latest.pt", map_location="cpu", weights_only=True)
print("Обучение сохранено. Итерация:", saved["iteration"], "Папка:", train_dir)
del saved

In [ ]:
from IPython.display import Image, Markdown, display

checkpoint = train_dir / "latest.pt"
run(
    "benchmark",
    "--config",
    "configs/benchmark_colab.json",
    "--checkpoint",
    checkpoint,
    "--out",
    output / "benchmark_gpu_50",
)
run(
    "arena",
    "--checkpoint",
    checkpoint,
    "--device",
    "cuda",
    "--method",
    "async",
    "--workers",
    16,
    "--games",
    100,
    "--simulations",
    128,
    "--out",
    output / "arena_gpu_50.json",
)
display(Markdown((output / "benchmark_gpu_50/REPORT.md").read_text()))
display(Image(filename=str(output / "benchmark_gpu_50/throughput.png")))

## Передача результатов для анализа
Сохраните название GPU, `environment.json`, конфигурации, исходные CSV, `summary.json`, графики и контрольную точку. Нельзя сравнивать скорость CPU крестиков-ноликов с GPU «Четыре в ряд» как ускорение одной системы: для CPU/GPU-сравнения используйте одну игру, сеть, веса и конфигурацию. Для этого повторите GPU-конфигурацию с `--device cpu` и другим выходным каталогом в том же Colab.


In [ ]:
# Упаковываем небольшой комплект измерений без больших весов.
measurements = Path("/content") / f"colab_measurements_{run_name}.zip"
with zipfile.ZipFile(measurements, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in output.rglob("*"):
        if path.is_file() and path.suffix in {".json", ".jsonl", ".csv", ".md", ".png"}:
            archive.write(path, path.relative_to(output))
shutil.copy2(measurements, output / measurements.name)
print("Архив измерений сохранён на Drive:", output / measurements.name)
files.download(str(measurements))
print("Контрольные точки сохранены на Drive:", output)